> **Public copy.** The outputs below are from the original run on the private database. After that run, product, vendor and system names and release numbers in the code, text and tables were replaced with the public codes used everywhere in this repository (Vendor A to E; releases such as 1.51 and "Vendor C 3.0"), so every number is exactly as computed. The database itself is not published; the anonymised data files are in `data/`.


# 09 Class weights or SMOTE? (step 4)

**The problem.** Only about 1 training row in 4 is a hotspot. Left alone, a model leans towards always saying "not a hotspot", and it would miss the rows QA cares about most.

**Two ways to fix it (the course mentor: "use SMOTE or class weights"):**
- **Class weights:** each real hotspot counts more while the model learns. No new rows are made.
- **SMOTE:** creates extra hotspot rows by blending two similar real hotspots into an in-between row. Component and vendor are names, not numbers, so the version for mixed data is used (SMOTE-NC). SMOTE is applied only to the training part of each check, never to the rows being checked.

**What is compared (five options x two models):**
1. No fix
2. Class weights "balanced": each class gets total rows / (2 x rows in the class), about 3 to 1 for hotspots
3. Class weights 2 to 1
4. Class weights 5 to 1
5. SMOTE-NC (no class weights)

Each option is run with the **decision tree** (the main model, depth 4 here) and **logistic regression**. Model settings are fixed at middle values in this step; step 5 tunes them.

**Training releases only.** Three time-ordered checks: learn from the older training releases, check on the next ones. The 16 test releases stay untouched until step 5. All training rows are used here, including 1.37 and Vendor C 3.0 (estimated effort); the with-and-without test is part of step 5.

In [1]:
import sqlite3, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd

# Works whether you run from the project folder or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
CLEAN = ROOT / 'data' / 'private' / 'hindsight_clean.sqlite'  # our cleaned tables
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore', category=FutureWarning)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score, matthews_corrcoef, recall_score, precision_score
from imblearn.pipeline import Pipeline            # applies SMOTE only while fitting, never when predicting
from imblearn.over_sampling import SMOTENC
from hs_features import FEATURES_NUM, FEATURES_CAT

with sqlite3.connect(CLEAN) as cc:
    fact = pd.read_sql('select * from fact_labelled', cc, parse_dates=['window_start', 'window_end'])
train = fact[fact.split == 'train'].copy()
X_COLS = FEATURES_NUM + FEATURES_CAT
print(f'training rows: {len(train)}, hotspots: {int(train.high_risk.sum())} ({train.high_risk.mean():.0%})')

training rows: 380, hotspots: 96 (25%)


## 1. The three time-ordered checks

In [2]:
order = train.groupby('release_key').window_start.min().sort_values()
n = len(order)
cuts = [int(n * q) for q in (0.55, 0.70, 0.85)] + [n]
folds = [(order.index[:cuts[i]], order.index[cuts[i]:cuts[i + 1]]) for i in range(3)]
for i, (fit_rel, check_rel) in enumerate(folds, start=1):
    a, b = train[train.release_key.isin(fit_rel)], train[train.release_key.isin(check_rel)]
    print(f'check {i}: learn from {len(fit_rel)} releases ({int(a.high_risk.sum())} hotspots), '
          f'check on {len(check_rel)} releases ({len(b)} rows, {int(b.high_risk.sum())} hotspots)')

check 1: learn from 19 releases (48 hotspots), check on 6 releases (66 rows, 15 hotspots)
check 2: learn from 25 releases (63 hotspots), check on 5 releases (45 rows, 9 hotspots)
check 3: learn from 30 releases (72 hotspots), check on 6 releases (84 rows, 24 hotspots)


## 2. The ten versions

The numbers go through the same steps in every version: empty history values are filled with the median of the rows being learned from, and the names (component, vendor) become yes/no columns. Logistic regression also puts the numbers on one scale.

In [3]:
NUM_IDX = list(range(len(FEATURES_NUM)))
CAT_IDX = [len(FEATURES_NUM), len(FEATURES_NUM) + 1]

def steps(model, scale, smote):
    """Fill gaps and code names -> (SMOTE-NC) -> one-hot names (and scale numbers) -> model."""
    fill = ColumnTransformer([('num', SimpleImputer(strategy='median'), FEATURES_NUM),
                              ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), FEATURES_CAT)])
    shape = ColumnTransformer([('num', StandardScaler() if scale else 'passthrough', NUM_IDX),
                               ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_IDX)])
    parts = [('fill', fill)]
    if smote:
        parts.append(('smote', SMOTENC(categorical_features=CAT_IDX, random_state=42)))
    return Pipeline(parts + [('shape', shape), ('model', model)])

OPTIONS = {
    '1. No fix': (None, False),
    '2. Class weights balanced (about 3 to 1)': ('balanced', False),
    '3. Class weights 2 to 1': ({0: 1, 1: 2}, False),
    '4. Class weights 5 to 1': ({0: 1, 1: 5}, False),
    '5. SMOTE-NC': (None, True),
}
MODELS = {
    'Decision tree (main model)': lambda w: DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, class_weight=w, random_state=42),
    'Logistic regression': lambda w: LogisticRegression(C=0.3, class_weight=w, max_iter=3000),
}
print(len(OPTIONS) * len(MODELS), 'versions')

10 versions


## 3. Results (average of the three checks)

| Measure | In plain words |
| --- | --- |
| PR-AUC | How well the version puts real hotspots at the top of its list (1 = perfect; guessing scores the share of hotspots) |
| Recall | Of the real hotspots, the share it flagged |
| Precision | Of the rows it flagged, the share that were real hotspots |
| MCC | One balanced score from -1 to 1 (0 = guessing), fair when hotspots are rare |

In [4]:
rows = []
for model_name, make in MODELS.items():
    for option, (weights, smote) in OPTIONS.items():
        for i, (fit_rel, check_rel) in enumerate(folds, start=1):
            a, b = train[train.release_key.isin(fit_rel)], train[train.release_key.isin(check_rel)]
            pipe = steps(make(weights), scale=model_name.startswith('Logistic'), smote=smote)
            pipe.fit(a[X_COLS], a.high_risk)
            p = pipe.predict_proba(b[X_COLS])[:, 1]
            flag = (p >= 0.5).astype(int)
            rows.append({'model': model_name, 'option': option, 'check': i,
                         'PR-AUC': average_precision_score(b.high_risk, p),
                         'Recall': recall_score(b.high_risk, flag, zero_division=0),
                         'Precision': precision_score(b.high_risk, flag, zero_division=0),
                         'MCC': matthews_corrcoef(b.high_risk, flag)})
results = pd.DataFrame(rows)
summary = results.groupby(['model', 'option'])[['PR-AUC', 'Recall', 'Precision', 'MCC']].mean().round(2)
summary

PR-AUC  Recall  Precision   MCC
model                      option                                                                   
Decision tree (main model) 1. No fix                                   0.57    0.37       0.61  0.36
                           2. Class weights balanced (about 3 to 1)    0.64    0.78       0.48  0.44
                           3. Class weights 2 to 1                     0.62    0.70       0.47  0.40
                           4. Class weights 5 to 1                     0.55    0.80       0.44  0.40
                           5. SMOTE-NC                                 0.62    0.78       0.51  0.48
Logistic regression        1. No fix                                   0.74    0.59       0.62  0.47
                           2. Class weights balanced (about 3 to 1)    0.73    0.86       0.51  0.51
                           3. Class weights 2 to 1                     0.74    0.82       0.55  0.52
                           4. Class weights 5 to 1                     0.72    0.90       0.50  0.52
                           5. SMOTE-NC                                 0.76    0.85       0.54  0.55

**How much each result moves between the three checks** (the lowest and highest MCC): with only about 60 to 70 rows in each check, small differences are within noise.

In [5]:
spread = results.groupby(['model', 'option']).MCC.agg(['min', 'max']).round(2)
spread.columns = ['MCC lowest check', 'MCC highest check']
spread

MCC lowest check  MCC highest check
model                      option                                                                       
Decision tree (main model) 1. No fix                                             0.09               0.53
                           2. Class weights balanced (about 3 to 1)              0.32               0.52
                           3. Class weights 2 to 1                               0.28               0.49
                           4. Class weights 5 to 1                               0.20               0.57
                           5. SMOTE-NC                                           0.42               0.55
Logistic regression        1. No fix                                             0.40               0.53
                           2. Class weights balanced (about 3 to 1)              0.40               0.59
                           3. Class weights 2 to 1                               0.38               0.61
                           4. Class weights 5 to 1                               0.42               0.59
                           5. SMOTE-NC                                           0.47               0.61

## 4. Reading the results

- **PR-AUC** (ranking) barely depends on the fix: weights and SMOTE mostly move the yes/no cut-off, not the order of the list.
- **Recall and precision** move against each other: a stronger weight catches more hotspots and raises more false alarms.
- **MCC** shows the balance of the two.

## 5. The choice: balanced class weights (Jagan, 24 Nov 2026)

Balanced class weights and SMOTE-NC are almost a tie. SMOTE is slightly ahead on MCC (tree 0.48 against 0.44; logistic regression 0.55 against 0.51), while balanced weights give the main model its best ranking (PR-AUC 0.64 against 0.62). Both gaps are smaller than the swing between the three checks (the tree's MCC moves between 0.32 and 0.52). So the simpler option wins: **class weights "balanced"**. It needs no invented rows and no extra library, and it was the course mentor's first suggestion. Stronger weights (5 to 1) catch slightly more hotspots but raise more false alarms, with no gain in MCC; weaker weights (2 to 1) catch fewer.

In step 5: logistic regression and the decision tree use `class_weight="balanced"`, the random forest uses `class_weight="balanced_subsample"` (the same idea, recalculated for each tree's sample), and the two baselines need no weights.

## 6. Save

In [6]:
with sqlite3.connect(CLEAN) as cc:
    summary.reset_index().to_sql('class_balance_results', cc, if_exists='replace', index=False)
print('saved class_balance_results', summary.shape)

saved class_balance_results (10, 4)
